In [1]:
import numpy as np

LINE = "=" * 72

# =====================================================================
# 5.1 非线性方程验证
# =====================================================================
print(LINE)
print("5.1 非线性方程验证")
print(LINE)

# --- 5.1-a: g(x)=cos(x) 误差比（对照 A1 表10） ---
print("\n[5.1-a] g(x)=cos(x), x0=0.5 的误差比（对照 A1 表10）")
x = 0.5
for _ in range(500):
    x = np.cos(x)
xstar = x
print(f"  不动点 x* = {xstar:.15f}")
print(f"  |g'(x*)| = |sin(x*)| = {abs(np.sin(xstar)):.6f}")

xs = [0.5]
for _ in range(5):
    xs.append(np.cos(xs[-1]))
errs = np.abs(np.array(xs) - xstar)
ratios = errs[1:] / errs[:-1]

print(f"  {'k':>2} {'x_k':>16} {'|e_k|':>14} {'ratio':>10}")
for k in range(len(xs)):
    r = f"{ratios[k-1]:.4f}" if k > 0 else "   ---"
    print(f"  {k:>2} {xs[k]:>16.10f} {errs[k]:>14.6e} {r:>10}")

print()
print("  A1 表10 给出比值：0.772, 0.690, 0.676, 0.671")
print(f"  本次复算比值：{ratios[0]:.3f}, {ratios[1]:.3f}, {ratios[2]:.3f}, {ratios[3]:.3f}")
print("  => 差异明显：表中比值列错误，应重算。")

# --- 5.1-b: 改写3 g(x)=x/2+1/x 误差（对照 A1 代码输出） ---
print("\n[5.1-b] 改写3 g(x)=x/2+1/x 的误差（对照 A1 代码输出）")
sqrt2 = np.sqrt(2)
xs = [1.0]
for _ in range(7):
    v = xs[-1]
    xs.append(v / 2 + 1 / v)
errs = np.abs(np.array(xs) - sqrt2)
print(f"  {'k':>2} {'x_k':>18} {'|e_k|':>14}")
for k in range(len(xs)):
    print(f"  {k:>2} {xs[k]:>18.12f} {errs[k]:>14.6e}")
print()
print(f"  三步后误差约 {errs[3]:.2e}，四步后约 {errs[4]:.2e}")
print("  正文称'三步内到达 1e-8' => 数量级错误；应改为'约四步到机器精度'。")

# =====================================================================
# 5.2 下降方向验证
# =====================================================================
print()
print(LINE)
print("5.2 下降方向验证")
print(LINE)

print("\n[5.2] A=diag(2,8) 条件数与稳定步长上界（A1 图5-12/5-13）")
A = np.diag([2.0, 8.0])
eig = np.linalg.eigvalsh(A)
kappa = eig.max() / eig.min()
print(f"  特征值 = {eig}")
print(f"  条件数 kappa = {kappa:.1f}")
print('  图5-12注写"条件数=20" => 与代码 diag(2,8) 不一致（应为 4）')
print()
print(f"  固定步长稳定上界 2/lambda_max = {2/eig.max():.3f}")
print('  图5-13注写"2/L=0.1, alpha=0.11 超过" => 与代码 alphas=[0.01,0.05,0.1,0.2,0.3] 不符')
print()
print(f"  {'alpha':>8} {'5步后||x||':>14} {'状态':>8}")
b = np.zeros(2)
x0 = np.array([1.0, 1.0])
for alpha in [0.01, 0.05, 0.1, 0.2, 0.3]:
    xv = x0.copy()
    for _ in range(5):
        g = A @ xv - b
        xv = xv - alpha * g
    mag = np.linalg.norm(xv)
    status = "发散" if mag > 10 else "收敛中"
    print(f"  {alpha:>8.2f} {mag:>14.4e} {status:>8}")

# =====================================================================
# 5.3 Newton 与拟 Newton 验证
# =====================================================================
print()
print(LINE)
print("5.3 Newton 与拟 Newton 验证")
print(LINE)

print("\n[5.3] A1 图5-17 测试函数 Hessian 正定性")
print("  f3(x,y) = exp(x) + x^2 + exp(y) + y^2")
print("  H = diag(exp(x)+2, exp(y)+2)")

def H3(v):
    xv, yv = v
    return np.array([[np.exp(xv) + 2.0, 0.0],
                     [0.0, np.exp(yv) + 2.0]])

pts = [(2.0, 2.0), (10.0, 10.0), (-5.0, -5.0), (20.0, -20.0)]
print(f"  {'point':>14} {'eig_min':>12} {'eig_max':>12} {'正定':>6}")
for p in pts:
    ev = np.linalg.eigvalsh(H3(p))
    print(f"  {str(p):>14} {ev.min():>12.4e} {ev.max():>12.4e} {str(ev.min() > 0):>6}")

print()
print("  结论：Hessian 处处正定（特征值 >= 2），纯 Newton 在该函数上")
print("        不会因 Hessian 不定而发散；图注'纯 Newton 发散'与测试函数不符。")
print("  建议：图注改为条件性表述，或换一个 Hessian 可能不定的测试函数。")

# =====================================================================
# 5.4 非线性最小二乘验证
# =====================================================================
print()
print(LINE)
print("5.4 非线性最小二乘验证")
print(LINE)

t = np.array([0.0, 1.0, 2.0])

# --- 5.4-a: 线性模型 GN 一步 ---
print("\n[5.4-a] 线性模型 GN 一步（A1 例5.4.2.1，初值 (0,0)）")
y_lin = np.array([1.0, 2.0, 4.0])
a0, b0 = 0.0, 0.0
r = y_lin - (a0 + b0 * t)
J = np.column_stack([-np.ones_like(t), -t])
JTJ = J.T @ J
JTr = J.T @ r
d = np.linalg.solve(JTJ, -JTr)
print(f"  r = {r}")
print(f"  J =\n{J}")
print(f"  J^T J =\n{JTJ}")
print(f"  J^T r = {JTr}")
print(f"  d = {d}")
print("  参考值 (5/6, 3/2) = (0.8333, 1.5000)")
print("  => 与解析解一致；但 A1 正文保留了'等等—符号相反'草稿段，需删除。")

# --- 5.4-b: 指数模型 GN 一步 ---
print("\n[5.4-b] 指数模型 GN 一步（A1 例5.4.2.2，初值 (1,0.5)）")
y_exp = np.array([1.0, 2.5, 6.0])
a0, b0 = 1.0, 0.5
e = np.exp(b0 * t)
r = y_exp - a0 * e
J = np.column_stack([-e, -a0 * t * e])
JTJ = J.T @ J
JTr = J.T @ r
d = np.linalg.solve(JTJ, -JTr)
print(f"  r = {r}")
print(f"  J =\n{J}")
print(f"  J^T J =\n{JTJ}")
print(f"  J^T r = {JTr}")
print(f"  d_GN = {d}")
print(f"  更新后 (a,b) = ({a0+d[0]:.4f}, {b0+d[1]:.4f})")
print("  A1 正文给 d ~ (0.90, 0.10) => 符号/数量级错误；应约 (-0.067, 0.633)。")

# --- 5.4-c: 指数模型 LM 一步 ---
print("\n[5.4-c] 指数模型 LM 一步 lambda=0.1（A1 例5.4.3.3）")
lam = 0.1
A_lm = JTJ + lam * np.eye(2)
d_lm = np.linalg.solve(A_lm, -JTr)
print(f"  d_LM = {d_lm}")
print(f"  更新后 (a,b) = ({a0+d_lm[0]:.4f}, {b0+d_lm[1]:.4f})")
print("  A1 正文给 d ~ (0.82, 0.15) => 数值错误；应约 (-0.043, 0.618)。")

# --- 5.4-d: GN 误差递推符号 ---
print("\n[5.4-d] GN 误差递推符号")
print("  正确形式：e_{k+1} = -(J^T J)^{-1} S e_k + O(||e_k||^2),")
print("           其中 S = sum_i r_i * grad^2 r_i")
print("  A1 深度透析中漏负号（写成了 +(J^T J)^{-1} S e_k），需修正。")

# =====================================================================
# 复现结论
# =====================================================================
print()
print(LINE)
print("复现结论与修正清单")
print(LINE)
print()
print("A1-G9 结论：数学主线基本正确，但不能直接通过。必须修改：")
print("  1. 5.1.2 表10比值列与代码输出'三步到 1e-8'错误；")
print("  2. 5.2.2 图5-12、图5-13 图注与代码参数不一致；")
print("  3. 5.3.1 把 C^3 写成优化 Newton 的必要条件，应改为'常用充分条件'；")
print("  4. 5.3.2 图5-17 图注与代码测试函数不符；")
print("  5. 5.4.2 例5.4.2.1 编辑残留；5.4.2 例5.4.2.2、5.4.3 例5.4.3.3 数值错误；")
print("  6. 5.4.2 深度透析 GN 误差递推漏负号；")
print("  7. 图5-23 图注置信带与代码不符；")
print("  8. 本章小结重复两次，需去重。")
print()
print("A2-G19 结论：主线、证明与数值例较严谨，未发现关键数值错误，小修后可通过。建议修改：")
print("  1. 5.3.3 算法补 eta 默认值；")
print("  2. 5.4.3 命题5.4.9 证明中 J(u,v)^T=0 改为 J[u;v]=0；")
print("  3. 5.4.1 加残差符号对照说明。")
print()
print("建议 A1 在修正后重跑实验 5.1-B、5.2-B、5.3-B、5.4-B/C，把上述替换后的数值")
print("与图注写回正文和 Notebook，再按 B 组模板提交正式复现表。")

5.1 非线性方程验证

[5.1-a] g(x)=cos(x), x0=0.5 的误差比（对照 A1 表10）
  不动点 x* = 0.739085133215161
  |g'(x*)| = |sin(x*)| = 0.673612
   k              x_k          |e_k|      ratio
   0     0.5000000000   2.390851e-01        ---
   1     0.8775825619   1.384974e-01     0.5793
   2     0.6390124942   1.000726e-01     0.7226
   3     0.8026851007   6.359997e-02     0.6355
   4     0.6947780268   4.430711e-02     0.6967
   5     0.7681958313   2.911070e-02     0.6570

  A1 表10 给出比值：0.772, 0.690, 0.676, 0.671
  本次复算比值：0.579, 0.723, 0.636, 0.697
  => 差异明显：表中比值列错误，应重算。

[5.1-b] 改写3 g(x)=x/2+1/x 的误差（对照 A1 代码输出）
   k                x_k          |e_k|
   0     1.000000000000   4.142136e-01
   1     1.500000000000   8.578644e-02
   2     1.416666666667   2.453104e-03
   3     1.414215686275   2.123901e-06
   4     1.414213562375   1.594724e-12
   5     1.414213562373   2.220446e-16
   6     1.414213562373   2.220446e-16
   7     1.414213562373   2.220446e-16

  三步后误差约 2.12e-06，四步后约 1.59e-12
  正文称'三步内到达 1e-8'